# Kindle — Data Exploration
Downloads the reading insights CSV from Google Drive and runs the extractor logic.

In [ ]:
import sys
import tempfile
from pathlib import Path

import gdown
import polars as pl
from dotenv import dotenv_values

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

env = dotenv_values(ROOT / '.env')
DRIVE_URL = env['DRIVE_SHARE_URL']

In [ ]:
tmp = tempfile.mkdtemp()
gdown.download_folder(url=DRIVE_URL, output=tmp, use_cookies=False, quiet=False)

kindle_dir = Path(tmp) / 'kindle'
csv_files = list(kindle_dir.glob('*.csv'))
print(f'Found {len(csv_files)} CSV(s):', [f.name for f in csv_files])

In [ ]:
from pipeline.extractors.kindle import _read_csv

frames = [_read_csv(f.read_bytes()) for f in csv_files]
df = pl.concat(frames)
print(f'{len(df)} rows, {df["date"].min()} → {df["date"].max()}')
df

In [ ]:
# Daily totals
df.group_by('date').agg(pl.col('value').sum().alias('total_minutes')).sort('date')

In [ ]:
# Per-book totals
df.group_by('category').agg(pl.col('value').sum().alias('total_minutes')).sort('total_minutes', descending=True)